# Deep Dive: Classes, Attributes, and Inheritance in Real-World Python

> **Reference**: Inspired by the core architecture taught in Real Python (*Object-Oriented Programming (OOP) in Python 3*), but elevated to a **production-grade enterprise domain** (Corporate Workforce, Compensation & Payroll Management) instead of toy animal/dog examples.

---

## Real Python Concepts Covered in this Guide:
1. **Classes vs. Instances**: Blueprint vs. concrete runtime objects in memory.
2. **Class Attributes vs. Instance Attributes**: State shared across all instances vs. state isolated per instance.
3. **Constructor (`__init__`) and `self`**: Explicit instance binding and initialization.
4. **Instance Methods & Business Invariants**: Encapsulating domain operations.
5. **Dunder Methods (`__str__` and `__repr__`)**: Providing clean user representation and robust developer logging/debugging.
6. **Inheritance & Hierarchy**: Base class (`Employee`) vs. specialized derived classes (`SoftwareEngineer`, `SalesExecutive`).
7. **Extending and Overriding with `super()`**: Reusing parent logic without duplication (DRY principle).
8. **Type Introspection & Polymorphism**: `isinstance()`, `issubclass()`, and processing polymorphic collections.


## 1. Class Blueprint, Class Attributes vs. Instance Attributes

In introductory tutorials, you typically see `Dog` with `species = 'Canis familiaris'` and instance attributes `name` and `age`.

In enterprise codebases, we use:
- **Class Attributes**: Global corporate constants, policies, or global registries/counters (e.g., `company_name`, `standard_work_hours`, `_total_employees`). Defined directly inside the class body, outside any method.
- **Instance Attributes**: Attributes bound to `self` in `__init__`, unique to each instantiated object (`emp_id`, `name`, `base_salary`, `department`).

In [ ]:
class Employee:
    # ── CLASS ATTRIBUTES (Shared across all instances) ──
    company_name: str = "TechCorp Global"
    standard_work_hours: int = 40
    base_annual_leave_days: int = 20
    _total_employees: int = 0  # Class-level counter

    def __init__(self, emp_id: str, name: str, base_salary: float, department: str):
        # ── INSTANCE ATTRIBUTES (Unique to each instance) ──
        self.emp_id = emp_id
        self.name = name
        self.base_salary = base_salary
        self.department = department
        self.remaining_leave_days = Employee.base_annual_leave_days
        
        # Increment class-level tracking counter
        Employee._total_employees += 1

    @classmethod
    def get_headcount(cls) -> int:
        """Returns the total headcount across the organization."""
        return cls._total_employees

# Instantiating instances (Objects)
emp1 = Employee("EMP-101", "Elena Vance", 85000.0, "Infrastructure")
emp2 = Employee("EMP-102", "Marcus Holloway", 92000.0, "Security")

print(f"emp1: {emp1.name} | Company: {emp1.company_name} | Salary: ${emp1.base_salary:,.2f}")
print(f"emp2: {emp2.name} | Company: {emp2.company_name} | Salary: ${emp2.base_salary:,.2f}")
print(f"Total Headcount: {Employee.get_headcount()}")

### Critical Production Gotcha: Attribute Shadowing

When accessing `emp1.company_name`, Python first checks `emp1.__dict__`. If not found, it traverses up to `Employee.__dict__`.

If you assign `emp1.company_name = 'TechCorp Labs'`, you **do not** modify the class attribute; you create an instance-level variable that "shadows" the class attribute on `emp1` only!

In [ ]:
# Modifying on an instance creates a shadow attribute on emp1 only
emp1.company_name = "TechCorp CyberLabs (Subsidiary)"

print("emp1.company_name:", emp1.company_name)            # Shadowed attribute
print("emp2.company_name:", emp2.company_name)            # Still reading the class attribute
print("Employee.company_name:", Employee.company_name)    # Class attribute remained untouched

# Deleting the instance attribute restores normal fallback to class attribute
del emp1.company_name
print("emp1 after del:", emp1.company_name)

## 2. Instance Methods and Business Invariants

Instance methods always receive `self` as their first parameter. They are where domain business rules, mutations, and validations live.

In [ ]:
class Employee:
    company_name: str = "TechCorp Global"
    base_annual_leave_days: int = 20

    def __init__(self, emp_id: str, name: str, base_salary: float, department: str):
        self.emp_id = emp_id
        self.name = name
        self.base_salary = base_salary
        self.department = department
        self.remaining_leave_days = self.base_annual_leave_days

    def calculate_monthly_pay(self) -> float:
        """Calculates monthly gross earnings."""
        return round(self.base_salary / 12, 2)

    def calculate_annual_bonus(self) -> float:
        """Baseline annual corporate bonus: 10% of base salary."""
        return round(self.base_salary * 0.10, 2)

    def request_leave(self, days: int) -> bool:
        """Applies business validation before deducting PTO days."""
        if days <= 0:
            raise ValueError("Leave request must be at least 1 day.")
        if days > self.remaining_leave_days:
            print(f"[REJECTED] {self.name} requested {days} days, but only {self.remaining_leave_days} days remain.")
            return False
        
        self.remaining_leave_days -= days
        print(f"[APPROVED] {self.name}: {days} days granted. Remaining: {self.remaining_leave_days}.")
        return True

worker = Employee("EMP-103", "Sophia Lin", 110000.0, "Data Platform")
print(f"Monthly pay: ${worker.calculate_monthly_pay():,.2f}")
print(f"Baseline bonus: ${worker.calculate_annual_bonus():,.2f}")
worker.request_leave(5)
worker.request_leave(18)  # Exceeds available balance

## 3. Dunder Methods: `__str__` and `__repr__` for Production Observability

In Real Python, `__str__` is highlighted because by default, printing an object outputs `<Employee object at 0x7fa12345>`.

In production software engineering, we implement **both** special methods:
- **`__str__`**: Human-friendly formatting for end-users, UI displays, and external reports.
- **`__repr__`**: Explicit, unambiguous representation for developers, logs, tracebacks, and interactive consoles (`eval(repr(obj))` goal).

In [ ]:
class Employee:
    company_name: str = "TechCorp Global"
    base_annual_leave_days: int = 20

    def __init__(self, emp_id: str, name: str, base_salary: float, department: str):
        self.emp_id = emp_id
        self.name = name
        self.base_salary = base_salary
        self.department = department

    def calculate_annual_bonus(self) -> float:
        return round(self.base_salary * 0.10, 2)

    def calculate_monthly_pay(self) -> float:
        return round(self.base_salary / 12, 2)

    def __str__(self) -> str:
        """User-friendly representation for reports and notifications."""
        return f"{self.name} ({self.emp_id}) - {self.department} | ${self.base_salary:,.2f}/yr"

    def __repr__(self) -> str:
        """Developer/debugger-friendly representation."""
        return (
            f"Employee(emp_id={self.emp_id!r}, name={self.name!r}, "
            f"base_salary={self.base_salary}, department={self.department!r})"
        )

dev = Employee("EMP-200", "Alex Mercer", 98000.0, "Backend")

# str() is called by print() and str()
print("str output  :", str(dev))

# repr() is called in logs, tracebacks, and inside collections
print("repr output :", repr(dev))
print("Inside list :", [dev])

## 4. Inheritance: Extending & Overriding with `super()`

In Real Python, the `Dog` class is inherited by `JackRussellTerrier` and `Bulldog`.

In our corporate domain, different roles share the core `Employee` contract, but have specialized domain capabilities and compensation models:
- **`SoftwareEngineer`**: Adds `tech_stack`, on-call rotation status, specialized deploy methods, and extra tech-certification bonus allowances.
- **`SalesExecutive`**: Adds `commission_rate` and `annual_sales_volume`, with bonuses driven purely by closed business revenue.

### Key Tools:
- **`super().__init__(...)`**: Delegates common attribute initialization to the parent class, enforcing DRY.
- **Method Overriding**: Re-implementing a parent method in a child class.
- **`super().method(...)`**: Invoking parent business logic within the overridden method to augment rather than duplicate logic.

In [ ]:
class SoftwareEngineer(Employee):
    """Specialized child class for technical software engineers."""

    def __init__(
        self,
        emp_id: str,
        name: str,
        base_salary: float,
        tech_stack: list[str],
        on_call: bool = False,
    ):
        # Delegate common initialization to parent Employee
        super().__init__(emp_id, name, base_salary, department="Engineering")
        self.tech_stack = tech_stack
        self.on_call = on_call

    # Child-specific operation
    def deploy_service(self, service_name: str, environment: str = "production") -> str:
        return f"[DEPLOY] {self.name} deployed '{service_name}' to {environment} using {self.tech_stack}."

    # Overriding and extending parent method using super()
    def calculate_annual_bonus(self) -> float:
        # Base 10% corporate bonus from Employee
        base_bonus = super().calculate_annual_bonus()
        
        # Engineering-specific bonuses: on-call stipend + tech cert allowances
        on_call_stipend = 5000.0 if self.on_call else 0.0
        tech_allowance = len(self.tech_stack) * 500.0
        
        return round(base_bonus + on_call_stipend + tech_allowance, 2)

    def __repr__(self) -> str:
        return (
            f"SoftwareEngineer(emp_id={self.emp_id!r}, name={self.name!r}, "
            f"base_salary={self.base_salary}, tech_stack={self.tech_stack!r}, on_call={self.on_call})"
        )


class SalesExecutive(Employee):
    """Specialized child class for revenue-generating sales professionals."""

    def __init__(
        self,
        emp_id: str,
        name: str,
        base_salary: float,
        commission_rate: float,
        annual_sales_volume: float = 0.0,
    ):
        super().__init__(emp_id, name, base_salary, department="Sales & Revenue")
        self.commission_rate = commission_rate
        self.annual_sales_volume = annual_sales_volume

    def record_sale(self, amount: float):
        self.annual_sales_volume += amount
        print(f"[DEAL CLOSED] {self.name} closed ${amount:,.2f}. Total volume: ${self.annual_sales_volume:,.2f}")

    # Overriding parent bonus completely: sales bonus is 100% commission-driven
    def calculate_annual_bonus(self) -> float:
        return round(self.annual_sales_volume * self.commission_rate, 2)

    def __repr__(self) -> str:
        return (
            f"SalesExecutive(emp_id={self.emp_id!r}, name={self.name!r}, "
            f"base_salary={self.base_salary}, commission_rate={self.commission_rate}, annual_sales_volume={self.annual_sales_volume})"
        )

## 5. Testing Inheritance in Practice

Let's instantiate both child classes and observe how inherited methods and overridden methods behave.

In [ ]:
lead_dev = SoftwareEngineer(
    emp_id="SWE-301",
    name="Sarah Connor",
    base_salary=130000.0,
    tech_stack=["Python", "FastAPI", "Kubernetes", "AWS"],
    on_call=True,
)

sales_lead = SalesExecutive(
    emp_id="SLS-401",
    name="Jordan Belfort",
    base_salary=60000.0,
    commission_rate=0.08,
    annual_sales_volume=1_250_000.0,
)

# 1. Inherited methods from base Employee class
print("Lead Dev Monthly Pay (Inherited):", f"${lead_dev.calculate_monthly_pay():,.2f}")
print("Sales Lead Monthly Pay (Inherited):", f"${sales_lead.calculate_monthly_pay():,.2f}")

# 2. Child-specific method
print(lead_dev.deploy_service("payment-gateway", "production"))

# 3. Overridden Bonus Calculation:
# Engineer: $13,000 (10% base) + $5,000 (on-call) + $2,000 (4 tech certs * $500) = $20,000
print(f"Lead Dev Bonus (Extended with super()): ${lead_dev.calculate_annual_bonus():,.2f}")

# Sales: 8% of $1,250,000 = $100,000
print(f"Sales Lead Bonus (Commission-based):     ${sales_lead.calculate_annual_bonus():,.2f}")

## 6. Type Introspection & Polymorphism: `isinstance()` and `issubclass()`

In Real Python, `isinstance(miles, Dog)` and `isinstance(miles, Bulldog)` are demonstrated.

In production software:
1. **Polymorphism**: We can process a diverse collection of `Employee` objects with a unified loop without caring about their specific subclass.
2. **`isinstance()`**: Used when specific security or business policies apply only to certain employee types (e.g., granting infrastructure deployment keys only to engineers).
3. **`issubclass()`**: Used in factory registries, plugin validation, and architecture testing.

In [ ]:
# Polymorphism: Global Payroll Processing
staff_directory: list[Employee] = [
    Employee("GEN-001", "Robert Taylor", 72000.0, "Human Resources"),
    lead_dev,
    sales_lead,
]

print("=== CORPORATE ANNUAL PAYROLL REPORT ===")
total_bonus_pool = 0.0

for employee in staff_directory:
    bonus = employee.calculate_annual_bonus()
    total_bonus_pool += bonus
    print(f"• {employee.name:<16} | Dept: {employee.department:<18} | Bonus: ${bonus:>10,.2f}")

print(f"\nTotal Corporate Bonus Pool Allocated: ${total_bonus_pool:,.2f}")

print("\n=== RUNTIME TYPE INTROSPECTION ===")
print(f"isinstance(lead_dev, Employee):          {isinstance(lead_dev, Employee)}")
print(f"isinstance(lead_dev, SoftwareEngineer):  {isinstance(lead_dev, SoftwareEngineer)}")
print(f"isinstance(lead_dev, SalesExecutive):    {isinstance(lead_dev, SalesExecutive)}")

print(f"\nissubclass(SoftwareEngineer, Employee):  {issubclass(SoftwareEngineer, Employee)}")
print(f"issubclass(Employee, SoftwareEngineer):  {issubclass(Employee, SoftwareEngineer)}")

## 7. Real Python Concept Mapping Summary

| Real Python Tutorial (Dog Park) | Enterprise Production Example | Key Design Concept |
| :--- | :--- | :--- |
| `class Dog:` | `class Employee:` | Blueprint defining encapsulated state and domain operations. |
| `species = "Canis familiaris"` | `company_name = "TechCorp Global"` | **Class Attribute**: Defined at class scope, shared across instances. |
| `def __init__(self, name, age):` | `def __init__(self, emp_id, name, ...):` | **Constructor**: Initializes per-instance state bound to `self`. |
| `def speak(self, sound):` | `def calculate_annual_bonus(self):` | **Instance Method**: Domain logic operating on instance data. |
| `def __str__(self):` | `__str__` and `__repr__` | **Dunder Methods**: User-facing string vs. developer-facing debugging/logging. |
| `class JackRussellTerrier(Dog):` | `class SoftwareEngineer(Employee):` | **Inheritance**: Subclass receives parent capabilities and defines specialized fields. |
| `super().speak(sound)` | `super().calculate_annual_bonus()` | **`super()`**: Calls parent method implementation to maintain DRY. |
| `isinstance(miles, Dog)` | `isinstance(lead_dev, Employee)` | **Type Introspection**: Runtime verification of class and inheritance hierarchy. |
